# Predictive Maintenance — Predicting Equipment Failure

This notebook demonstrates **predictive AI** for industrial equipment maintenance:

| Maintenance strategy | Cost | Risk |
|---|---|---|
| **Reactive** — fix when it breaks | Low upfront | Unplanned downtime, collateral damage |
| **Scheduled** — maintain on calendar | Medium | Over-maintenance of healthy equipment |
| **Predictive** — maintain when data says so | Optimized | Minimal downtime, no waste |

We use sensor data from 8 CNC machines to predict: **will this machine fail within 7 days?**

### Data
- 6 months of IoT sensor readings (temperature, vibration, pressure, RPM, power, acoustics)
- 4 readings per day per machine (every 6 hours)
- Binary target: `failure_within_7d`

## 1. Load and explore data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

df = pd.read_csv('data/equipment_sensors.csv', parse_dates=['timestamp'])
print(f"Dataset: {df.shape[0]} readings, {df.shape[1]} features")
print(f"Machines: {df['machine_id'].nunique()}")
print(f"Period: {df['timestamp'].min().date()} to {df['timestamp'].max().date()}")
print(f"\nTarget distribution:")
print(df['failure_within_7d'].value_counts().to_string())
print(f"\nFailure rate: {df['failure_within_7d'].mean():.1%}")
df.describe().round(1)

## 2. Visualize sensor patterns

Let's look at one machine and see how sensor readings change before a failure event.

In [ ]:
machine = df[df['machine_id'] == 'CNC-001'].copy()
failure_periods = machine[machine['failure_within_7d'] == 1]['timestamp']

sensors = ['temperature_c', 'vibration_mm_s', 'pressure_bar', 'acoustic_db']
labels = ['Temperature (°C)', 'Vibration (mm/s)', 'Pressure (bar)', 'Acoustic (dB)']
colors = ['#C9190B', '#0066CC', '#3E8635', '#F0AB00']

fig, axes = plt.subplots(len(sensors), 1, figsize=(14, 10), sharex=True)
fig.suptitle('CNC-001 — Sensor readings over 6 months', fontsize=14, fontweight='bold')

for ax, sensor, label, color in zip(axes, sensors, labels, colors):
    ax.plot(machine['timestamp'], machine[sensor], color=color, linewidth=0.6, alpha=0.8)
    # Rolling average
    rolling = machine.set_index('timestamp')[sensor].rolling('3D').mean()
    ax.plot(rolling.index, rolling.values, color=color, linewidth=1.8, label='3-day avg')
    # Highlight failure windows
    for ts in failure_periods:
        ax.axvspan(ts - pd.Timedelta(days=7), ts, alpha=0.15, color='red')
    ax.set_ylabel(label, fontsize=10)
    ax.grid(True, alpha=0.3)

axes[0].legend(['Raw', '3-day avg', 'Failure window'], loc='upper right', fontsize=9)
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

print("Red shading = 7-day window before failure. Notice how temperature and vibration")
print("rise, while pressure drops — the signature of impending failure.")

## 3. Feature engineering

Raw sensor values alone are not enough. We engineer features that capture **trends** and **anomalies**:

In [ ]:
def engineer_features(df):
    """Create features that capture degradation patterns."""
    result = df.copy()
    sensors = ['temperature_c', 'vibration_mm_s', 'pressure_bar', 'rpm', 'power_kw', 'acoustic_db']
    
    for machine_id in result['machine_id'].unique():
        mask = result['machine_id'] == machine_id
        machine_data = result.loc[mask].set_index('timestamp')
        
        for sensor in sensors:
            series = machine_data[sensor]
            # Rolling statistics (24h = 4 readings)
            result.loc[mask, f'{sensor}_mean_24h'] = series.rolling(4, min_periods=1).mean().values
            result.loc[mask, f'{sensor}_std_24h'] = series.rolling(4, min_periods=1).std().fillna(0).values
            # Rate of change
            result.loc[mask, f'{sensor}_delta'] = series.diff().fillna(0).values
            # 3-day rolling mean for trend
            result.loc[mask, f'{sensor}_mean_3d'] = series.rolling(12, min_periods=1).mean().values
    
    # Interaction features
    result['temp_vibration_product'] = result['temperature_c'] * result['vibration_mm_s']
    result['power_per_rpm'] = result['power_kw'] / result['rpm'].clip(lower=1)
    result['vibration_to_pressure'] = result['vibration_mm_s'] / result['pressure_bar'].clip(lower=1)
    
    return result

df_feat = engineer_features(df)

feature_cols = [c for c in df_feat.columns if c not in 
                ['timestamp', 'machine_id', 'failure_within_7d']]

print(f"Features: {len(feature_cols)} (was {len(sensors)} raw sensors)")
print(f"\nNew features include:")
print("  - 24h rolling mean & std for each sensor")
print("  - Rate of change (delta) for each sensor") 
print("  - 3-day rolling mean for trend detection")
print("  - Cross-sensor interactions (temp*vibration, power/rpm, vibration/pressure)")

## 4. Train and compare models

We use a **time-based split** (not random!) — train on first 4 months, test on last 2.  
This simulates production: we never peek into the future.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, roc_auc_score, 
                             precision_recall_curve, roc_curve, f1_score)

# Time-based split: train on Jul-Oct, test on Nov-Dec
split_date = pd.Timestamp('2025-11-01')
train = df_feat[df_feat['timestamp'] < split_date]
test = df_feat[df_feat['timestamp'] >= split_date]

X_train = train[feature_cols].fillna(0)
y_train = train['failure_within_7d']
X_test = test[feature_cols].fillna(0)
y_test = test['failure_within_7d']

print(f"Train: {len(train)} samples ({y_train.mean():.1%} positive)")
print(f"Test:  {len(test)} samples ({y_test.mean():.1%} positive)")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10, 
                                            class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=5, 
                                                    learning_rate=0.1, random_state=42),
}

results = {}
for name, model in models.items():
    if 'Logistic' in name:
        model.fit(X_train_scaled, y_train)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_prob = model.predict_proba(X_test)[:, 1]
    
    y_pred = (y_prob >= 0.5).astype(int)
    results[name] = {
        'model': model,
        'y_prob': y_prob,
        'y_pred': y_pred,
        'auc': roc_auc_score(y_test, y_prob),
        'f1': f1_score(y_test, y_pred),
    }
    print(f"\n{'='*60}")
    print(f"{name} — AUC: {results[name]['auc']:.3f}, F1: {results[name]['f1']:.3f}")
    print(f"{'='*60}")
    print(classification_report(y_test, y_pred, target_names=['Normal', 'Pre-failure']))

## 5. ROC and Precision-Recall curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors_m = ['#0066CC', '#3E8635', '#C9190B']

for (name, res), color in zip(results.items(), colors_m):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax1.plot(fpr, tpr, color=color, linewidth=2, label=f"{name} (AUC={res['auc']:.3f})")
    
    prec, rec, _ = precision_recall_curve(y_test, res['y_prob'])
    ax2.plot(rec, prec, color=color, linewidth=2, label=f"{name} (F1={res['f1']:.3f})")

ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Feature importance — what drives predictions?

In [ ]:
best_model_name = max(results, key=lambda k: results[k]['auc'])
best = results[best_model_name]
print(f"Best model: {best_model_name} (AUC={best['auc']:.3f})\n")

if hasattr(best['model'], 'feature_importances_'):
    importances = pd.Series(best['model'].feature_importances_, index=feature_cols)
    top_15 = importances.nlargest(15)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    top_15.sort_values().plot(kind='barh', ax=ax, color='#0066CC', edgecolor='white')
    ax.set_title(f'Top 15 Features — {best_model_name}', fontweight='bold')
    ax.set_xlabel('Importance')
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("Key insight: rolling statistics and cross-sensor features dominate —")
    print("the TREND matters more than the raw value.")

## 7. Business impact simulation

What does this model save in practice?

In [ ]:
# Cost assumptions
COST_UNPLANNED_DOWNTIME = 15_000   # EUR per event — emergency repair + lost production
COST_PLANNED_MAINTENANCE = 2_000   # EUR per event — scheduled, prepared
COST_FALSE_ALARM = 500             # EUR — unnecessary inspection

y_pred_best = best['y_pred']

tp = ((y_pred_best == 1) & (y_test == 1)).sum()  # Caught failures
fn = ((y_pred_best == 0) & (y_test == 1)).sum()  # Missed failures
fp = ((y_pred_best == 1) & (y_test == 0)).sum()  # False alarms

cost_no_model = (tp + fn) * COST_UNPLANNED_DOWNTIME
cost_with_model = (tp * COST_PLANNED_MAINTENANCE + 
                   fn * COST_UNPLANNED_DOWNTIME + 
                   fp * COST_FALSE_ALARM)
savings = cost_no_model - cost_with_model

print(f"{'Test period results':=^50}")
print(f"  Failures caught (true positives):  {tp:>4}")
print(f"  Failures missed (false negatives): {fn:>4}")
print(f"  False alarms (false positives):    {fp:>4}")
print()
print(f"{'Cost comparison':=^50}")
print(f"  Without model (all unplanned):  EUR {cost_no_model:>10,}")
print(f"  With model (predictive):        EUR {cost_with_model:>10,}")
print(f"  {'─'*42}")
print(f"  Savings:                        EUR {savings:>10,}")
print(f"  Reduction:                          {savings/cost_no_model:.0%}")

# Visualization
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(['Without model\n(reactive)', 'With model\n(predictive)'],
               [cost_no_model, cost_with_model],
               color=['#C9190B', '#3E8635'], edgecolor='white', height=0.5)
ax.bar_label(bars, fmt='EUR {:,.0f}', padding=5, fontweight='bold')
ax.set_xlabel('Maintenance cost (EUR) — test period')
ax.set_title('Business Impact: Predictive vs Reactive Maintenance', fontweight='bold')
ax.set_xlim(0, cost_no_model * 1.3)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Timeline — predictions for one machine

In [ ]:
test_machine = test[test['machine_id'] == 'CNC-002'].copy()
if hasattr(best['model'], 'predict_proba'):
    if 'Logistic' in best_model_name:
        X_m = scaler.transform(test_machine[feature_cols].fillna(0))
    else:
        X_m = test_machine[feature_cols].fillna(0)
    test_machine['failure_prob'] = best['model'].predict_proba(X_m)[:, 1]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.plot(test_machine['timestamp'], test_machine['temperature_c'], 
         color='#C9190B', alpha=0.5, linewidth=0.8, label='Temperature')
ax1.plot(test_machine['timestamp'], test_machine['vibration_mm_s'] * 30 + 40, 
         color='#0066CC', alpha=0.5, linewidth=0.8, label='Vibration (scaled)')
ax1.set_ylabel('Sensor value')
ax1.legend(loc='upper left', fontsize=9)
ax1.set_title('CNC-002 — November–December 2025', fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.fill_between(test_machine['timestamp'], test_machine['failure_prob'], 
                 alpha=0.4, color='#C9190B')
ax2.plot(test_machine['timestamp'], test_machine['failure_prob'], 
         color='#C9190B', linewidth=1.5)
ax2.axhline(y=0.5, color='black', linestyle='--', alpha=0.5, label='Decision threshold')

# Mark actual failure windows
failure_mask = test_machine['failure_within_7d'] == 1
if failure_mask.any():
    for ts in test_machine.loc[failure_mask, 'timestamp']:
        ax2.axvline(x=ts, color='red', alpha=0.1, linewidth=3)

ax2.set_ylabel('Failure probability')
ax2.set_xlabel('Date')
ax2.set_ylim(-0.05, 1.05)
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Top: raw sensor readings. Bottom: model's predicted failure probability.")
print("Red shading = actual pre-failure periods. The model spikes BEFORE the failure window.")

## Key Takeaways

| Aspect | Details |
|---|---|
| **Problem** | Predict equipment failure 7 days ahead from IoT sensors |
| **Data** | 5,760 readings, 8 machines, 6 months, 12 raw features → 39 engineered |
| **Best model** | Gradient Boosting or Random Forest with ~0.95+ AUC |
| **Key features** | Rolling statistics and cross-sensor interactions beat raw values |
| **Business value** | 60–80% reduction in maintenance costs |

### From notebook to production on OpenShift AI
1. **Register model** in Model Registry (version, metadata, lineage)
2. **Serve model** via KServe — real-time scoring or batch
3. **Monitor** with Observe & monitor dashboard — data drift, prediction quality
4. **Automate retraining** with Pipelines when drift is detected
5. **Govern** — who trained it, on what data, who approved deployment